# 基于 K 近邻（KNN）的乳腺癌良恶性分类

**课程**：机器学习  
**负责算法**：K 近邻 (K-Nearest Neighbors)  
**数据集**：Breast Cancer Wisconsin (Diagnostic) Data Set  
**来源**：https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

## 一、导入所需库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)
import warnings
warnings.filterwarnings('ignore')

# 直接指定字体文件路径，绕过所有字体缓存问题
ZH = fm.FontProperties(fname='C:/Windows/Fonts/msyh.ttc')
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
print("所有库导入成功！")

## 二、数据加载与探索

In [ ]:
# 加载威斯康星乳腺癌数据集
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='diagnosis')

print(f"数据集样本数：{X.shape[0]}")
print(f"特征数量：{X.shape[1]}")
print(f"\n类别分布：")
print(f"  恶性 (Malignant, 0)：{(y == 0).sum()} 例 ({(y == 0).sum()/len(y)*100:.1f}%)")
print(f"  良性 (Benign, 1)：{(y == 1).sum()} 例 ({(y == 1).sum()/len(y)*100:.1f}%)")
print(f"\n特征列表（共 {len(data.feature_names)} 个）：")
for i, name in enumerate(data.feature_names):
    print(f"  {i+1:2d}. {name}")

In [ ]:
# 数据概览
print("前 5 行数据：")
display(X.head())

print("\n基本统计描述：")
display(X.describe().round(3))

print(f"\n缺失值总数：{X.isnull().sum().sum()}")

In [ ]:
# 特征分布可视化
key_features = ['mean radius', 'mean texture', 'mean area', 'mean smoothness',
                'mean concavity', 'mean concave points', 'worst radius', 'worst area']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.hist(X.loc[y == 0, feat], bins=25, alpha=0.6, label='恶性', color='#e74c3c', edgecolor='white')
    ax.hist(X.loc[y == 1, feat], bins=25, alpha=0.6, label='良性', color='#3498db', edgecolor='white')
    ax.set_title(f'{feat}', fontsize=11, fontproperties=ZH)
    ax.set_xlabel(feat, fontproperties=ZH)
    ax.set_ylabel('频数', fontproperties=ZH)
    ax.legend(prop=fm.FontProperties(fname='C:/Windows/Fonts/msyh.ttc', size=9))

plt.suptitle('良性 vs 恶性肿瘤各特征分布', fontsize=15, y=1.01, fontproperties=ZH)
plt.tight_layout()
plt.show()

## 三、数据预处理

KNN 是基于距离度量的算法，特征标准化至关重要——否则量级大的特征会主导距离计算。

In [ ]:
# 划分训练集 (80%) 和测试集 (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集：{X_train.shape[0]} 样本")
print(f"测试集：{X_test.shape[0]} 样本")
print(f"\n训练集分布：恶性 {(y_train == 0).sum()} 例, 良性 {(y_train == 1).sum()} 例")
print(f"测试集分布：恶性 {(y_test == 0).sum()} 例, 良性 {(y_test == 1).sum()} 例")

In [ ]:
# 标准化：KNN 的关键步骤
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("标准化完成！")
print(f"标准化后训练集均值 ≈ 0：{np.mean(np.abs(X_train_scaled.mean(axis=0))):.6f}")
print(f"标准化后训练集标准差 ≈ 1：{np.mean(X_train_scaled.std(axis=0)):.6f}")

## 四、KNN 模型训练与评估

### 4.1 基础 KNN（k=5，默认欧氏距离）

In [ ]:
def evaluate_knn(model, X_train, X_test, y_train, y_test, model_name):
    """训练 KNN 模型并输出评估结果"""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')

    print(f"{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  准确率 (Accuracy):   {acc:.4f}")
    print(f"  精确率 (Precision):  {prec:.4f}")
    print(f"  召回率 (Recall):     {rec:.4f}")
    print(f"  F1 分数:             {f1:.4f}")
    print(f"  5 折交叉验证均值:    {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'cm': cm, 'y_proba': y_proba, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()}

# 基础 KNN
knn_base = KNeighborsClassifier(n_neighbors=5)
res_base = evaluate_knn(knn_base, X_train_scaled, X_test_scaled, y_train, y_test, 'KNN (k=5, 欧氏距离)')

print("\n详细分类报告：")
print(classification_report(y_test, knn_base.predict(X_test_scaled), target_names=data.target_names))

### 4.2 超参数调优：寻找最优 k 值

k 值过小会导致过拟合（对噪声敏感），k 值过大会导致分类边界过于平滑。通过 GridSearchCV 寻找最佳 k。

In [ ]:
# GridSearchCV 搜索最佳 k 值
param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1=曼哈顿距离, 2=欧氏距离
}

knn_grid = KNeighborsClassifier()
grid_search = GridSearchCV(knn_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print(f"最佳参数：{grid_search.best_params_}")
print(f"最佳交叉验证准确率：{grid_search.best_score_:.4f}")

In [ ]:
# 不同 k 值下的准确率变化
k_range = range(1, 21)
cv_scores_k = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores_k.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(k_range, cv_scores_k, 'o-', color='#3498db', linewidth=2, markersize=6)
best_k = k_range[np.argmax(cv_scores_k)]
plt.axvline(x=best_k, color='#e74c3c', linestyle='--', alpha=0.7,
            label=f'最佳 k = {best_k} (准确率 = {max(cv_scores_k):.4f})')
plt.xlabel('k 值 (近邻数量)', fontsize=12, fontproperties=ZH)
plt.ylabel('5 折交叉验证准确率', fontsize=12, fontproperties=ZH)
plt.title('不同 k 值下 KNN 模型准确率变化', fontsize=14, fontproperties=ZH)
plt.xticks(k_range)
plt.legend(prop=fm.FontProperties(fname='C:/Windows/Fonts/msyh.ttc', size=10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 用最优参数重新训练
knn_best = KNeighborsClassifier(**grid_search.best_params_)
res_best = evaluate_knn(knn_best, X_train_scaled, X_test_scaled, y_train, y_test,
                        f'KNN (k={grid_search.best_params_["n_neighbors"]}, 优化后)')

print("\n详细分类报告：")
print(classification_report(y_test, knn_best.predict(X_test_scaled), target_names=data.target_names))

### 4.3 混淆矩阵可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 基础 KNN
sns.heatmap(res_base['cm'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=data.target_names, yticklabels=data.target_names,
            annot_kws={'size': 14})
axes[0].set_title('基础 KNN (k=5)', fontsize=13, fontproperties=ZH)
axes[0].set_xlabel('预测标签', fontproperties=ZH)
axes[0].set_ylabel('真实标签', fontproperties=ZH)

# 优化后 KNN
sns.heatmap(res_best['cm'], annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=data.target_names, yticklabels=data.target_names,
            annot_kws={'size': 14})
axes[1].set_title(f'优化后 KNN (k={grid_search.best_params_["n_neighbors"]})', fontsize=13, fontproperties=ZH)
axes[1].set_xlabel('预测标签', fontproperties=ZH)
axes[1].set_ylabel('真实标签', fontproperties=ZH)

plt.suptitle('KNN 混淆矩阵对比', fontsize=15, fontproperties=ZH)
plt.tight_layout()
plt.show()

### 4.4 ROC 曲线

In [ ]:
plt.figure(figsize=(8, 6))

for name, proba, color in [
    ('基础 KNN (k=5)', res_base['y_proba'], '#3498db'),
    (f'优化后 KNN (k={grid_search.best_params_["n_neighbors"]})', res_best['y_proba'], '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='随机分类器')
plt.xlabel('假阳性率 (False Positive Rate)', fontsize=12, fontproperties=ZH)
plt.ylabel('真阳性率 (True Positive Rate)', fontsize=12, fontproperties=ZH)
plt.title('KNN 模型 ROC 曲线', fontsize=14, fontproperties=ZH)
plt.legend(prop=fm.FontProperties(fname='C:/Windows/Fonts/msyh.ttc', size=10))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 五、结论

### 结果汇总

| 指标 | 基础 KNN (k=5) | 优化后 KNN (k=4) |
|------|:----------:|:--------:|
| 准确率 | 0.9561 | 0.9561 |
| 精确率 | 0.9589 | 0.9589 |
| 召回率 | 0.9722 | 0.9722 |
| F1 分数 | 0.9655 | 0.9655 |
| 5 折 CV 均值 | 0.9670 (±0.0209) | 0.9714 (±0.0237) |

### 关键发现

1. **标准化是 KNN 的关键前提**——KNN 依赖距离度量，量纲不一致会严重扭曲结果。
2. **GridSearchCV 找到的最佳 k 值为 4**，权重策略为 `uniform`（等权投票），距离度量为曼哈顿距离（L1）。
3. 优化后的 KNN（k=4）在测试集上取得了 **95.61%** 的准确率，说明 KNN 在该数据集上有良好的分类能力。超参数调优主要提升了交叉验证稳定性（从 0.9670 提升至 0.9714）。
4. KNN 的缺点：预测时需要遍历所有训练样本计算距离，效率较低；且对特征尺度敏感，必须配合标准化使用。